In [1]:
import re
import pandas as pd
from collections import Counter
from datasets import load_dataset


/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
ds_train = load_dataset("Riksarkivet/swener_1800", split="train")

#  gliner2's word splitter 
from gliner2.processor import WhitespaceTokenSplitter
sp = WhitespaceTokenSplitter()
split = lambda t: [(s, e) for _, s, e in sp(t)]
split("Test ord.")  # smoke test

tot, ok = Counter(), Counter()
bad = []  # misaligned examples
for doc_id, text, types, starts, ends in zip(
    ds_train["id"], ds_train["text"], ds_train["entity_types"],
    ds_train["entity_start_chars"], ds_train["entity_end_chars"]
):
    words = split(text)
    ws, we = {s for s, _ in words}, {e for _, e in words}
    for t, s, e in zip(types, starts, ends):
        tot[t] += 1
        if s in ws and e in we:
            ok[t] += 1
        else:
            bad.append((doc_id, t, s, e, "start" if s not in ws else "end" if e not in we else "both"))

res = pd.DataFrame({"total": tot, "aligned": ok}).fillna(0).astype(int)
res["aligned_%"] = (res.aligned / res.total * 100).round(1)
display(res.sort_values("total", ascending=False))
print(f"overall aligned: {sum(ok.values())}/{sum(tot.values())} = {sum(ok.values())/sum(tot.values()):.1%}")

# look at possible misaligned spans
text_by_id = dict(zip(ds_train["id"], ds_train["text"]))
for doc_id, t, s, e, side in bad[:12]:
    tx = text_by_id[doc_id]
    print(f"{t:9s} ({side}) ...{tx[max(0,s-20):s]}[[{tx[s:e]}]]{tx[e:e+20]}...".replace("\n", " "))

,total,aligned,aligned_%
PER,9387,9387,100.0
LOC,4552,4552,100.0
OCC,3708,3708,100.0
TME-DATE,2768,2768,100.0
MSR-MON,1476,1476,100.0
ORG-INST,1390,1390,100.0
TME-TIME,961,961,100.0
WRK,780,780,100.0
EVN,290,290,100.0
ORG-COMP,255,255,100.0


overall aligned: 26341/26341 = 100.0%


In [14]:
import random
import re
from collections import defaultdict
from datasets import load_dataset
from gliner2.processor import WhitespaceTokenSplitter
from gliner2.training.data import InputExample, TrainingDataset

CHUNK, SEED, DEV_FRAC = 128, 42, 0.1

LABELS = {
    "PER": "person", "LOC": "location", "OCC": "occupation",
    "TME-DATE": "date", "TME-TIME": "time expression", "TME-INTRV": "time interval",
    "MSR-MON": "money amount", "MSR-WEI": "weight", "MSR-VOL": "volume",
    "MSR-LEN": "length", "MSR-DIST": "distance", "MSR-AREA": "area",
    "MSR-OTH": "other measurement",
    "ORG-INST": "institution", "ORG-COMP": "company", "ORG-OTH": "other organization",
    "EVN": "event", "WRK": "work of art or product name", "SYMP": "symptom",
}

def make_cuts(T, text, gold, n):
    """Sentence-packed chunk boundaries (token indices), never cutting inside a gold span."""
    L = len(T)
    if L == 0:
        return [0, 0]
    cand = set()
    for i in range(L - 1):
        w = text[T[i][1]:T[i][2]]
        nxt = text[T[i + 1][1]:T[i + 1][2]]
        if (w in (".", "!", "?") and nxt[:1].isupper()) or "\n" in text[T[i][2]:T[i + 1][1]]:
            cand.add(i + 1)
    cuts, a = [0], 0
    while a < L:
        b = a + n
        if b >= L:
            break
        ok = [c for c in cand if a < c <= b]
        if ok:
            b = max(ok)
        moved = True
        while moved:                          # extend until the edge is outside every gold span
            moved = False
            for i, j, *_ in gold:
                if i < b <= j:
                    b, moved = j + 1, True
        cuts.append(b)
        a = b
    cuts.append(L)
    return cuts

def to_examples(rows):
    out = []
    for text, types, starts, ends in rows:
        T = list(sp(text, lower=True))
        if not T:
            continue
        s2i = {s: i for i, (_, s, _) in enumerate(T)}
        e2i = {e: i for i, (_, _, e) in enumerate(T)}
        gold = [(s2i[s], e2i[e], t, text[s:e]) for t, s, e in zip(types, starts, ends)
                if s in s2i and e in e2i]
        cuts = make_cuts(T, text, gold, CHUNK)
        for a, b in zip(cuts[:-1], cuts[1:]):
            if b <= a:
                continue
            ents = {name: [] for name in LABELS.values()}      # all 19 labels, empty by default
            for i, j, t, m in gold:
                if i >= a and j < b and m not in ents[LABELS[t]]:
                    ents[LABELS[t]].append(m)
            out.append(InputExample(text=text[T[a][1]:T[b - 1][2]], entities=ents))
    return out

# split by DOCUMENT first, then chunk (same seed/fraction as before -> same split)
rows = list(zip(ds_train["text"], ds_train["entity_types"],
                ds_train["entity_start_chars"], ds_train["entity_end_chars"]))
idx = list(range(len(rows)))
random.Random(SEED).shuffle(idx)
n_dev = int(len(rows) * DEV_FRAC)
dev_rows = [rows[i] for i in idx[:n_dev]]
trn_rows = [rows[i] for i in idx[n_dev:]]

train_ds = TrainingDataset(to_examples(trn_rows))
dev_ds = TrainingDataset(to_examples(dev_rows))
print(f"docs: train {len(trn_rows)}, dev {len(dev_rows)} | "
      f"chunks: train {len(train_ds.examples)}, dev {len(dev_ds.examples)}")

for name, d in [("train", train_ds), ("dev", dev_ds)]:
    r = d.validate(raise_on_error=False)
    print(name, {k: v for k, v in r.items() if not isinstance(v, (list, dict))})
train_ds.print_stats()
dev_ds.print_stats()
train_ds.save("train.jsonl"); dev_ds.save("dev.jsonl")

docs: train 610, dev 67 | chunks: train 4365, dev 325
train {'valid': 4365, 'invalid': 0, 'total': 4365}
dev {'valid': 325, 'invalid': 0, 'total': 325}

GLiNER2 Training Dataset Statistics
Total examples: 4365

Text lengths: min=1, max=1225, mean=527.0

Task Distribution:
  entities_only: 4365 (100.0%)

Entity Types (22331 total mentions):
  person: 7564
  location: 3957
  occupation: 3211
  date: 2511
  money amount: 1210
  institution: 1205
  time expression: 865
  work of art or product name: 676
  event: 250
  company: 219
  volume: 153
  other organization: 137
  symptom: 126
  weight: 68
  length: 52
  other measurement: 47
  time interval: 28
  distance: 28
  area: 24


GLiNER2 Training Dataset Statistics
Total examples: 325

Text lengths: min=7, max=871, mean=517.5

Task Distribution:
  entities_only: 325 (100.0%)

Entity Types (2005 total mentions):
  person: 663
  location: 347
  occupation: 251
  date: 184
  money amount: 176
  institution: 124
  work of art or product name:

In [15]:
import time, torch
from gliner2 import AutoExtractor
from gliner2.training.trainer import ExtractorTrainer, TrainingConfig

assert len(train_ds.examples) > 4000 and len(dev_ds.examples) > 300, "build train_ds / dev_ds first"
print("mps available:", torch.backends.mps.is_available())

model = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1", map_location="mps")

config = TrainingConfig(
    output_dir="./swener_ft",
    experiment_name="swener_gliner25_multi",
    num_epochs=5,
    batch_size=4,
    gradient_accumulation_steps=4,
    eval_batch_size=8,
    encoder_lr=1e-5,
    task_lr=5e-4,
    warmup_ratio=0.1,
    scheduler_type="cosine",
    fp16=False, bf16=False,
    eval_strategy="steps",
    eval_steps=270,                # about one epoch
    save_best=True,
    save_total_limit=2,            # each checkpoint is ~1.1 GB
    early_stopping=True,
    early_stopping_patience=2,
    num_workers=0,
    logging_steps=25,
    seed=42,
)

t0 = time.time()
results = ExtractorTrainer(model, config).train(train_data=train_ds, eval_data=dev_ds)
print(f"done in {(time.time() - t0) / 60:.0f} min")

keep = ("step", "eval_loss", "eval_proposal_oracle_recall",
        "eval_recall_length_9_plus", "eval_absent_query_false_positive_rate")
for h in results["eval_metrics_history"]:
    print({k: round(v, 3) for k, v in h.items() if k in keep})

mps available: True


/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/gliner2/models/boundary/model.py:2101: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)
/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/gliner2/models/boundary/model.py:1147: RuntimeWarning: Encoder rejected attn_implementation='sdpa'; falling back to 'eager' (DebertaV2Model does not support an attention implementation through torch.nn.functional.scaled_dot_product_attention yet. Please request the support for this architecture: https://github.com/huggingface/transformers/issues/28005. If you believe this error is a bug, please open an issue in Transformers GitHub repos

done in 138 min
{'eval_loss': 440.073, 'step': 270, 'eval_proposal_oracle_recall': 0.901, 'eval_recall_length_9_plus': 0.758, 'eval_absent_query_false_positive_rate': 0.037}
{'eval_loss': 406.198, 'step': 540, 'eval_proposal_oracle_recall': 0.919, 'eval_recall_length_9_plus': 0.788, 'eval_absent_query_false_positive_rate': 0.042}
{'eval_loss': 396.621, 'step': 810, 'eval_proposal_oracle_recall': 0.923, 'eval_recall_length_9_plus': 0.818, 'eval_absent_query_false_positive_rate': 0.044}
{'eval_loss': 385.464, 'step': 1080, 'eval_proposal_oracle_recall': 0.921, 'eval_recall_length_9_plus': 0.848, 'eval_absent_query_false_positive_rate': 0.049}
{'eval_loss': 381.59, 'step': 1350, 'eval_proposal_oracle_recall': 0.92, 'eval_recall_length_9_plus': 0.848, 'eval_absent_query_false_positive_rate': 0.044}


In [17]:
from collections import defaultdict
import pandas as pd

def evaluate(ds, preds, labels=None, merge_org=None, name=""):
    """
    ds     : eval split of Riksarkivet/swener_1800
    preds  : iterable of (doc_id, start, end, label); char offsets into the original text
    labels : labels to score, e.g. ["PER", "LOC", "ORG"] for ScandiNER.
             None = every label in gt (after ORG merge). Predictions outside this set are dropped.
    """
    gt = set()
    if merge_org is None:  
        merge_org = labels is not None and "ORG" in labels
    norm = lambda l: "ORG" if merge_org and l.startswith("ORG") else l
    for doc_id, types, starts, ends in zip(
            ds["id"], ds["entity_types"], ds["entity_start_chars"], ds["entity_end_chars"]
        ):
            for t, s, e in zip(types, starts, ends):
                gt.add((doc_id, s, e, norm(t)))

    pred = {(d, s, e, norm(l)) for d, s, e, l in preds}

    keep = set(labels) if labels else {x[3] for x in gt}
    gt = {x for x in gt if x[3] in keep}
    pred = {x for x in pred if x[3] in keep}

    def index(spans):
        d = defaultdict(list)
        for doc, s, e, lab in spans:
            d[doc].append((s, e, lab))
        return d

    gi, pi = index(gt), index(pred)

    def hit(item, idx):  # overlaps at least one span with the same label
        doc, s, e, lab = item
        return any(s < e2 and s2 < e and lab2 == lab for s2, e2, lab2 in idx[doc])

    def prf(tp_p, n_p, tp_r, n_g):
        p = tp_p / n_p if n_p else 0
        r = tp_r / n_g if n_g else 0
        f = 2 * p * r / (p + r) if p + r else 0
        return round(p, 3), round(r, 3), round(f, 3)

    rows = []
    for lab in sorted(keep) + ["micro"]:
        g = {x for x in gt if lab == "micro" or x[3] == lab}
        p = {x for x in pred if lab == "micro" or x[3] == lab}
        strict = prf(len(g & p), len(p), len(g & p), len(g))
        relaxed = prf(sum(hit(x, gi) for x in p), len(p),
                      sum(hit(x, pi) for x in g), len(g))
        rows.append((lab, len(g), len(p), *strict, *relaxed))

    res = pd.DataFrame(rows, columns=["label", "gt", "pred",
                                      "P_strict", "R_strict", "F1_strict",
                                      "P_relax", "R_relax", "F1_relax"])
    per_label = res[res.label != "micro"]
    print(f"== {name} | labels: {len(keep)} | "
          f"macro F1 strict {per_label.F1_strict.mean():.3f} / relaxed {per_label.F1_relax.mean():.3f}")
    display(res)
    return res

In [18]:
from tqdm.auto import tqdm
from gliner2 import AutoExtractor

NAME_TO_CODE = {v: k for k, v in LABELS.items()}
dev_docs = ds_train.select(idx[:n_dev])      # the same 67 held-out documents

def predict(m, docs, chunk_size):
    overlap = 32 if chunk_size <= 128 else 64
    out = set()
    for doc_id, text in tqdm(zip(docs["id"], docs["text"]), total=len(docs), leave=False):
        r = m.extract_entities_long(text, list(LABELS.values()),
                                    chunk_size=chunk_size, chunk_overlap=overlap,
                                    include_spans=True)
        for name, ents in r["entities"].items():
            for e in ents:
                out.add((doc_id, e["start"], e["end"], NAME_TO_CODE[name]))
    return out

models = {
    "zero-shot":  AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1", map_location="mps"),
    "fine-tuned": AutoExtractor.from_pretrained("./swener_ft/best", map_location="mps"),
}

results = {}
for mname, m in models.items():
    for cs in (128, 384):
        preds_dev = predict(m, dev_docs, cs)
        results[(mname, cs)] = evaluate(dev_docs, preds_dev, name=f"{mname} | chunk {cs} | dev")

summary = pd.DataFrame({f"{k[0]} @{k[1]}": r[r.label == "micro"].iloc[0, 3:] for k, r in results.items()}).T
display(summary)

The tokenizer you are loading from './swener_ft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
                                               

== zero-shot | chunk 128 | dev | labels: 19 | macro F1 strict 0.229 / relaxed 0.297


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,30,8,0.125,0.033,0.053,0.125,0.033,0.053
1,LOC,366,762,0.322,0.669,0.434,0.440,0.828,0.574
2,MSR-AREA,1,2,0.000,0.000,0.000,0.000,0.000,0.000
3,MSR-DIST,6,12,0.500,1.000,0.667,0.500,1.000,0.667
4,MSR-LEN,11,16,0.188,0.273,0.222,0.250,0.364,0.296
5,MSR-MON,196,154,0.474,0.372,0.417,0.714,0.551,0.622
6,MSR-OTH,3,4,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,10,16,0.125,0.200,0.154,0.125,0.200,0.154
8,MSR-WEI,11,5,0.400,0.182,0.250,0.400,0.182,0.250
9,OCC,273,177,0.542,0.352,0.427,0.582,0.377,0.458


== zero-shot | chunk 384 | dev | labels: 19 | macro F1 strict 0.242 / relaxed 0.319


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,30,10,0.000,0.000,0.000,0.000,0.000,0.000
1,LOC,366,635,0.384,0.667,0.488,0.515,0.828,0.635
2,MSR-AREA,1,3,0.000,0.000,0.000,0.000,0.000,0.000
3,MSR-DIST,6,6,1.000,1.000,1.000,1.000,1.000,1.000
4,MSR-LEN,11,16,0.125,0.182,0.148,0.125,0.182,0.148
5,MSR-MON,196,138,0.514,0.362,0.425,0.783,0.541,0.640
6,MSR-OTH,3,0,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,10,9,0.222,0.200,0.211,0.444,0.400,0.421
8,MSR-WEI,11,2,0.500,0.091,0.154,0.500,0.091,0.154
9,OCC,273,144,0.618,0.326,0.427,0.667,0.348,0.457


== fine-tuned | chunk 128 | dev | labels: 19 | macro F1 strict 0.563 / relaxed 0.629


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,30,22,0.409,0.300,0.346,0.409,0.300,0.346
1,LOC,366,394,0.685,0.738,0.711,0.802,0.850,0.825
2,MSR-AREA,1,1,1.000,1.000,1.000,1.000,1.000,1.000
3,MSR-DIST,6,5,1.000,0.833,0.909,1.000,0.833,0.909
4,MSR-LEN,11,10,0.900,0.818,0.857,0.900,0.818,0.857
5,MSR-MON,196,197,0.645,0.648,0.646,0.944,0.939,0.941
6,MSR-OTH,3,0,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,10,10,0.500,0.500,0.500,0.500,0.500,0.500
8,MSR-WEI,11,2,1.000,0.182,0.308,1.000,0.182,0.308
9,OCC,273,311,0.678,0.773,0.723,0.711,0.806,0.755


== fine-tuned | chunk 384 | dev | labels: 19 | macro F1 strict 0.502 / relaxed 0.574


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,30,19,0.368,0.233,0.286,0.421,0.267,0.327
1,LOC,366,322,0.764,0.672,0.715,0.879,0.765,0.818
2,MSR-AREA,1,1,1.000,1.000,1.000,1.000,1.000,1.000
3,MSR-DIST,6,5,1.000,0.833,0.909,1.000,0.833,0.909
4,MSR-LEN,11,10,0.900,0.818,0.857,0.900,0.818,0.857
5,MSR-MON,196,154,0.675,0.531,0.594,0.974,0.765,0.857
6,MSR-OTH,3,0,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,10,6,0.500,0.300,0.375,0.667,0.400,0.500
8,MSR-WEI,11,1,1.000,0.091,0.167,1.000,0.091,0.167
9,OCC,273,265,0.709,0.689,0.699,0.732,0.707,0.719


,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
zero-shot @128,0.382,0.414,0.397,0.579,0.606,0.592
zero-shot @384,0.44,0.378,0.407,0.672,0.559,0.61
fine-tuned @128,0.713,0.728,0.721,0.837,0.859,0.848
fine-tuned @384,0.733,0.604,0.662,0.867,0.725,0.79


In [32]:
import pickle
from datasets import load_dataset

LABELS = {
    "PER": "person", "LOC": "location", "OCC": "occupation",
    "TME-DATE": "date", "TME-TIME": "time expression", "TME-INTRV": "time interval",
    "MSR-MON": "money amount", "MSR-WEI": "weight", "MSR-VOL": "volume",
    "MSR-LEN": "length", "MSR-DIST": "distance", "MSR-AREA": "area",
    "MSR-OTH": "other measurement",
    "ORG-INST": "institution", "ORG-COMP": "company", "ORG-OTH": "other organization",
    "EVN": "event", "WRK": "work of art or product name", "SYMP": "symptom",
}

NAME_TO_CODE = {v: k for k, v in LABELS.items()}

ds_eval = load_dataset("Riksarkivet/swener_1800", split="eval")

final = {
    "fine-tuned @128": ("fine-tuned", 128),
    "zero-shot @128":  ("zero-shot", 128),
}
eval_preds = {}
for label, (mname, cs) in final.items():
    eval_preds[label] = predict(models[mname], ds_eval, cs)
    evaluate(ds_eval, eval_preds[label], name=f"{label} | EVAL | all labels")

pickle.dump(eval_preds, open("eval_preds.pkl", "wb"))   

== fine-tuned @128 | EVAL | all labels | labels: 19 | macro F1 strict 0.469 / relaxed 0.593


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,60,36,0.444,0.267,0.333,0.611,0.367,0.458
1,LOC,1350,1331,0.730,0.719,0.724,0.881,0.865,0.873
2,MSR-AREA,12,4,0.750,0.250,0.375,0.750,0.250,0.375
3,MSR-DIST,8,8,0.500,0.500,0.500,0.750,0.750,0.750
4,MSR-LEN,9,3,0.667,0.222,0.333,0.667,0.222,0.333
5,MSR-MON,471,462,0.732,0.718,0.725,0.950,0.932,0.941
6,MSR-OTH,20,0,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,47,38,0.579,0.468,0.518,0.605,0.489,0.541
8,MSR-WEI,15,10,0.600,0.400,0.480,0.800,0.533,0.640
9,OCC,1065,1135,0.801,0.854,0.826,0.832,0.887,0.859


== zero-shot @128 | EVAL | all labels | labels: 19 | macro F1 strict 0.199 / relaxed 0.316


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,60,17,0.059,0.017,0.026,0.059,0.017,0.026
1,LOC,1350,2478,0.342,0.627,0.443,0.571,0.895,0.697
2,MSR-AREA,12,5,0.600,0.250,0.353,0.600,0.250,0.353
3,MSR-DIST,8,10,0.200,0.250,0.222,0.500,0.625,0.556
4,MSR-LEN,9,23,0.043,0.111,0.062,0.174,0.556,0.265
5,MSR-MON,471,486,0.521,0.537,0.529,0.681,0.688,0.684
6,MSR-OTH,20,5,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,47,56,0.214,0.255,0.233,0.286,0.340,0.311
8,MSR-WEI,15,10,0.400,0.267,0.320,0.500,0.333,0.400
9,OCC,1065,660,0.641,0.397,0.490,0.691,0.427,0.528


In [22]:
import pickle
import pandas as pd
from collections import defaultdict, Counter

preds = pickle.load(open("eval_preds.pkl", "rb"))["fine-tuned @128"]

gold = set()
for doc_id, types, starts, ends in zip(ds_eval["id"], ds_eval["entity_types"],
                                       ds_eval["entity_start_chars"], ds_eval["entity_end_chars"]):
    for t, s, e in zip(types, starts, ends):
        gold.add((doc_id, s, e, t))

pred_by_doc = defaultdict(list)
for d, s, e, l in preds:
    pred_by_doc[d].append((s, e, l))

rows, confusions = [], Counter()
for d, s, e, lab in gold:
    if (d, s, e, lab) in preds:
        out = "exact"
    else:
        ov = [(min(e, e2) - max(s, s2), l2) for s2, e2, l2 in pred_by_doc[d] if s < e2 and s2 < e]
        if any(l2 == lab for _, l2 in ov):
            out = "boundary"
        elif ov:
            out = "wrong type"
            confusions[(lab, max(ov)[1])] += 1   # label of the prediction overlapping most
        else:
            out = "missed"
    rows.append((lab, out))

cols = ["exact", "boundary", "wrong type", "missed"]
df = pd.DataFrame(rows, columns=["gold", "outcome"])
tab = pd.crosstab(df.gold, df.outcome).reindex(columns=cols, fill_value=0)
tab["total"] = tab.sum(axis=1)
pct = (tab[cols].div(tab["total"], axis=0) * 100).round(0).astype(int).add_suffix(" %")
display(tab.join(pct).sort_values("total", ascending=False))

print("\nTop confusions (gold -> predicted):")
print(pd.Series(confusions).sort_values(ascending=False).head(15).to_string())

outcome,exact,boundary,wrong type,missed,total,exact %,boundary %,wrong type %,missed %
gold,,,,,,,,,
PER,1861,423,73,131,2488,75,17,3,5
LOC,971,197,73,109,1350,72,15,5,8
OCC,909,36,28,92,1065,85,3,3,9
TME-DATE,609,91,25,52,777,78,12,3,7
MSR-MON,338,101,6,26,471,72,21,1,6
ORG-INST,257,18,47,43,365,70,5,13,12
TME-TIME,164,37,0,14,215,76,17,0,7
WRK,60,33,30,37,160,38,21,19,23
ORG-COMP,17,10,28,14,69,25,14,41,20



Top confusions (gold -> predicted):
PER       OCC         50
LOC       PER         50
ORG-INST  LOC         34
OCC       PER         24
ORG-COMP  LOC         17
PER       LOC         15
TME-DATE  TME-TIME    15
WRK       LOC         13
LOC       ORG-COMP    10
ORG-INST  ORG-OTH      9
ORG-OTH   ORG-INST     8
ORG-COMP  PER          7
WRK       PER          5
ORG-OTH   LOC          5
TME-DATE  EVN          5


In [ ]:
import random

text_by_id = dict(zip(ds_eval["id"], ds_eval["text"]))

def outcome(d, s, e, lab):
    if (d, s, e, lab) in preds:
        return "exact"
    ov = [(s2, e2, l2) for s2, e2, l2 in pred_by_doc[d] if s < e2 and s2 < e]
    if any(l2 == lab for _, _, l2 in ov):
        return "boundary"
    return "wrong type" if ov else "missed"

def show(label, kind, n=8, seed=0):
    items = [g for g in gold if g[3] == label and outcome(*g) == kind]
    print(f"\n=== gold {label} | {kind} | {len(items)} cases ===")
    random.seed(seed)
    for d, s, e, lab in random.sample(items, min(n, len(items))):
        t = text_by_id[d]
        print(f"...{t[max(0, s-30):s]}[[{t[s:e]}]]{t[e:e+30]}...".replace("\n", " "))
        for s2, e2, l2 in pred_by_doc[d]:
            if s < e2 and s2 < e:
                print(f"     pred {l2}: {t[s2:e2]!r}")

# look at examples, cjust change the tag + kind of error here
show("PER", "boundary")
show("PER", "wrong type")
show("WRK", "missed")


=== gold PER | boundary | 423 cases ===
... Rörande wådjad ſak emellan : [[Gabriel Olsſon i Waßmolöſa]] , k : de , ſamt Lorentz Pette...
     pred LOC: 'Waßmolöſa'
     pred PER: 'Gabriel Olsſon'
...dska och svenska mynt , hafva [[förutnämnde konstaplar Andrén och Lindemark]] den 27 Februari undersökt skä...
     pred PER: 'konstaplar Andrén och Lindemark'
     pred OCC: 'konstaplar'
...No 170 .   son och [[Johannes Eriksson]] . Poliskonstapeln No 27 Tagel...
     pred PER: 'Eriksson'
...rne C. G. Rydin och C. Bäck , [[Gördelmakarne L. Carlsſon och W. A. Breuning]] ſamt i öfrigt af ſtadens inne...
     pred PER: 'Gördelmakarne L. Carlsſon'
     pred PER: 'W. A. Breuning'
     pred OCC: 'Gördelmakarne'
...lust som förorsakats honom af [[förre Bokhållaren Charles Kindberg]] , han återkallade sin den 29 ...
     pred OCC: 'Bokhållaren'
     pred PER: 'Charles Kindberg'
... de derefter , i sällskap med [[Arbetskarlarne Carl Ludvig Johansson och Sundberg]] , begifvit sig till Åkaren Ol...


In [34]:
LABELS = {
    "PER": "person", "LOC": "location", "ORG": "organisation",
}

model = AutoExtractor.from_pretrained("./swener_ft/best", map_location="mps")

NAME_TO_CODE = {v: k for k, v in LABELS.items()}

CHUNK, OVERLAP = 128, 32  

gl_preds_bios = set()  # (doc_id, start, end, dataset_label)
for doc_id, text in tqdm(zip(ds_eval["id"], ds_eval["text"]), total=len(ds_eval)):
    out = model.extract_entities_long(
        text,
        list(LABELS.values()),
        chunk_size=CHUNK,
        chunk_overlap=OVERLAP,
        include_spans=True,
    )
    for name, ents in out["entities"].items():
        for e in ents:
            gl_preds_bios.add((doc_id, e["start"], e["end"], NAME_TO_CODE[name]))

print(len(gl_preds_bios))
display(pd.Series([p[3] for p in gl_preds_bios]).value_counts())

# score on PER/LOC/ORG (the evaluate() cell must be defined in this notebook)
evaluate(ds_eval, gl_preds_bios, labels=["PER", "LOC", "ORG"], name="fine-tuned best | PER/LOC/ORG labels")

The tokenizer you are loading from './swener_ft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/gliner2/models/boundary/model.py:1147: RuntimeWarning: Encoder rejected attn_implementation='sdpa'; falling back to 'eager' (DebertaV2Model does not support an attention implementation through torch.nn.functional.scaled_dot_product_attention yet. Please request the support for this architecture: https://github.com/huggingface/transformers/issues/28005. If you believe this error is a bug, please open an issue in Transformers GitHub repository and load your model with the argument `attn_implementation="eager"` meanwhile. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_im

4268


PER    2439
LOC    1360
ORG     469
Name: count, dtype: int64

== fine-tuned best | PER/LOC/ORG labels | labels: 3 | macro F1 strict 0.718 / relaxed 0.842


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,LOC,1350,1360,0.721,0.726,0.723,0.877,0.873,0.875
1,ORG,455,469,0.635,0.655,0.645,0.693,0.716,0.705
2,PER,2488,2439,0.793,0.778,0.785,0.949,0.946,0.947
3,micro,4293,4268,0.753,0.748,0.751,0.898,0.899,0.898


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,LOC,1350,1360,0.721,0.726,0.723,0.877,0.873,0.875
1,ORG,455,469,0.635,0.655,0.645,0.693,0.716,0.705
2,PER,2488,2439,0.793,0.778,0.785,0.949,0.946,0.947
3,micro,4293,4268,0.753,0.748,0.751,0.898,0.899,0.898


In [35]:
sum(len(r[1]) for r in trn_rows)

24148